# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [8]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [9]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)

Root project: c:\Users\valer\Desktop\Analiza Datelor Complexe\17. AI Avansat\echochamber-project-team3
Gemini key found: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [10]:
student_id = "student_01"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [11]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [12]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [ ]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(10)

source_channel
RecorderRomania                   12177
turcescu111                        5019
georgesimionoficial                3669
CălinGeorgescu-CanalulOficial      3460
@CălinGeorgescu-CanalulOficial     2557
TuDecizi-s3g                        647
StareaNatiei                        623
AltcevacuAdrianArtene               363
roxindaniel                         305
otvdirect                           304
Name: count, dtype: int64

In [34]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [35]:
sample_df = df.sample(10, random_state=42).copy()
sample_df[["source_channel", "text"]]

,source_channel,text
23002,CălinGeorgescu-CanalulOficial,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,E dureros.. e crunt.. simt vinovatie si recuno...
9644,RecorderRomania,Cite dosare ați judecat și nu ați recuperat ni...
23843,turcescu111,"Totul duce către: Noua Ordine Mondială, pentru..."
11605,RecorderRomania,"Un hot corupt arogant si nesimtit, caruia nime..."
15486,RecorderRomania,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ..."
7767,RecorderRomania,Vă mai dau niște firme din Galați care au alți...


Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [36]:
SYSTEM_PROMPT = """
Esti un analist specializat in analiza discursului politic romanesc online, in special comentarii de pe retele sociale (YouTube).

Sarcina ta este sa analizezi un singur comentariu in limba romana si sa extragi structurat mai multe axe de analiza, inclusiv "bula discursiva" din care pare sa provina comentariul.

Reguli importante:
- Raspunzi DOAR cu JSON valid, fara text suplimentar inainte sau dupa.
- Folosesti EXCLUSIV valorile din taxonomiile permise pentru campurile cu enum.
- Daca nu poti determina o valoare in mod rezonabil, folosesti "unclear".
- Nu inventezi informatie care nu apare in comentariu.
- Distingi clar intre sentiment (cum suna comentariul in general) si stance (pozitionarea fata de tinta) - un comentariu poate fi negativ ca ton dar pro-tinta.
- Toate valorile text liber sunt in limba romana, dar fara diacritice.
"""

USER_PROMPT_TEMPLATE = """
Citeste urmatorul comentariu politic in limba romana si identifica:

1. target: persoana, partidul sau entitatea principala despre care vorbeste comentariul (text scurt, ex: "Calin Georgescu", "George Simion", "AUR", "sistemul"). Daca nu exista o tinta identificabila, foloseste null.

2. stance: pozitionarea autorului fata de target. Valori permise:
   - "pro" - sustine / apara tinta
   - "contra" - critica / ataca tinta
   - "neutru" - mentioneaza fara pozitionare clara
   - "unclear" - nu se poate determina

3. sentiment: incarcatura emotionala generala. Valori permise:
   - "pozitiv", "negativ", "mixt", "neutru"

4. tone: registrul dominant. Valori permise:
   - "factual", "ironic", "agresiv", "emotional", "religios", "conspirativ", "umoristic"

5. topic: tema principala. Valori permise:
   - "alegeri", "justitie", "suveranitate", "religie", "economie", "media", "ordine_publica", "personal_atac", "personal_sustinere", "altul"

6. interpretation_problem: principala dificultate de interpretare. Valori permise:
   - "none" - comentariul e clar
   - "sarcasm" - pare una, e alta
   - "ambiguitate" - sensul nu e clar
   - "tinte_multiple" - vorbeste despre mai multe entitati
   - "sentiment_vs_stance" - tonul si pozitionarea difera

7. rhetoric_type: figura retorica dominanta. Valori permise:
   - "none", "apel_la_emotie", "apel_la_autoritate", "whataboutism", "dezumanizare", "idolatrizare", "victimizare"

8. bubble: bula discursiva din care pare sa provina comentariul. Valori permise:
   - "personalist-salvator" - exprima speranta intr-un lider providential / salvator al natiunii (ex: idolatrizare Calin Georgescu, "Doamne ajuta ca avem un om ca el", apel la traditie si nostalgie nationala)
   - "anti-sistem" - critica generala a clasei politice si institutiilor ("toti sunt la fel", "ne-au furat tara", revolta fara o solutie clara)
   - "conspirationist" - vede planuri ascunse, manipulari, forte oculte ("se vede clar ca", "sistemul controleaza tot", referinte la elite globale)
   - "pro-european" - apara valorile UE, statul de drept, institutiile democratice, criticile sunt nuantate si argumentate
   - "altul" - nu se incadreaza clar in cele 4 bule
   - "unclear" - nu se poate determina

Important:
- Returneaza JSON valid cu EXACT aceste 8 chei: target, stance, sentiment, tone, topic, interpretation_problem, rhetoric_type, bubble.
- Adauga si un camp "justification" cu o propozitie scurta (max 20 cuvinte) care explica in romana alegerile principale, in special asocierea cu bula.

Format de raspuns asteptat:
{{
  "target": "...",
  "stance": "...",
  "sentiment": "...",
  "tone": "...",
  "topic": "...",
  "interpretation_problem": "...",
  "rhetoric_type": "...",
  "bubble": "...",
  "justification": "..."
}}

Comentariu:
<<< {comment_text} >>>
"""

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [46]:
from openai import OpenAI
client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [47]:
def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [53]:
n_comments = 10
sample_for_prompt = sample_df.head(n_comments)

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....,"```json\n{\n ""target"": ""Calin Georgescu"",\n ..."
1,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...,"```json\n{\n ""target"": ""Calin Georgescu"",\n ..."
2,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...,"```json\n{\n ""target"": ""autoritatile"",\n ""st..."
3,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...,"```json\n{\n ""target"": ""unclear"",\n ""stance""..."
4,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...,"```json\n{\n ""target"": ""sistemul"",\n ""stance..."
5,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Cite dosare ați judecat și nu ați recuperat ni...,"```json\n{\n ""target"": ""sistemul judiciar"",\n..."
6,turcescu111,"Frică, foame, sărăcie","Totul duce către: Noua Ordine Mondială, pentru...","```json\n{\n ""target"": ""noua ordine mondiala""..."
7,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"Un hot corupt arogant si nesimtit, caruia nime...","```json\n{\n ""target"": ""un hot corupt"",\n ""s..."
8,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un pr...,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ...","```json\n{\n ""target"": ""Crin Antonescu"",\n ""..."
9,RecorderRomania,Investigație din interiorul rețelei care te în...,Vă mai dau niște firme din Galați care au alți...,"```json\n{\n ""target"": ""firme din Galati"",\n ..."


# 9. Verificam rezultatele

In [54]:
results_df.model_output[0]

'```json\n{\n  "target": "Calin Georgescu",\n  "stance": "pro",\n  "sentiment": "mixt",\n  "tone": "ironic",\n  "topic": "personal_sustinere",\n  "interpretation_problem": "sentiment_vs_stance",\n  "rhetoric_type": "none",\n  "bubble": "personalist-salvator",\n  "justification": "Comentariul sustine ironic un lider, dorindu-i succes, specific bulei personalist-salvatoare."\n}\n```'

In [55]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [56]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....,Calin Georgescu,pro,mixt,ironic,personal_sustinere,sentiment_vs_stance,
1,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...,Calin Georgescu,pro,pozitiv,emotional,personal_sustinere,none,
2,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...,autoritatile,contra,negativ,emotional,ordine_publica,none,
3,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...,unclear,neutru,negativ,emotional,altul,none,
4,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...,sistemul,contra,negativ,emotional,altul,none,
5,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Cite dosare ați judecat și nu ați recuperat ni...,sistemul judiciar,contra,negativ,agresiv,justitie,none,
6,turcescu111,"Frică, foame, sărăcie","Totul duce către: Noua Ordine Mondială, pentru...",noua ordine mondiala,contra,negativ,conspirativ,altul,none,
7,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,"Un hot corupt arogant si nesimtit, caruia nime...",un hot corupt,contra,negativ,agresiv,personal_atac,none,
8,RecorderRomania,Reportaj Recorder în noaptea alegerilor: Un pr...,"4:30 și încă 1% rămas pentru Crin Alcoolescu, ...",Crin Antonescu,contra,negativ,ironic,alegeri,none,
9,RecorderRomania,Investigație din interiorul rețelei care te în...,Vă mai dau niște firme din Galați care au alți...,firme din Galati,neutru,neutru,factual,altul,none,


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [57]:
csv_output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.csv"
csv_output_file.parent.mkdir(parents=True, exist_ok=True)

parsed_df.to_csv(csv_output_file, index=False, encoding="utf-8-sig")
print("CSV salvat la:", csv_output_file)

CSV salvat la: c:\Users\valer\Desktop\Analiza Datelor Complexe\17. AI Avansat\echochamber-project-team3\outputs\student_01_prompt_outputs.csv


# 11. Reflecție asupra promptului

### Unde a funcționat bine promptul?

Promptul a identificat corect ținta principală pe **9 din 10 comentarii** și a plasat corect tema (`topic`) în categoriile predefinite în majoritatea cazurilor. Comentariile clar polarizate cu o singură țintă identificabilă — ex: rândul 5 ("Cite dosare ați judecat..." → `target: sistemul judiciar`, `stance: contra`, `topic: justitie`), rândul 7 ("Un hot corupt arogant" → `target: un hot corupt`, `stance: contra`, `tone: agresiv`, `topic: personal_atac`) sau rândul 8 ("4:30 și încă 1% rămas pentru Crin Alcoolescu" → `target: Crin Antonescu`, `tone: ironic`, `topic: alegeri`) — au fost clasificate consistent cu intuiția umană.

### Unde a eșuat?

- **Rândul 0** — "Multă sănătate dl. Președinte Călin Georgescu. Noi mergem până la capăt și vă dorim să fiți acceptat măcar președinte de scară de bloc. Și să nu fugiți în Dubai..." — Modelul a marcat `stance: pro`, dar comentariul este **clar sarcastic / contra**: poreclele "președinte de scară de bloc" și aluzia la "fuga în Dubai" sunt ironice. Modelul a fost păcălit de formulele de politețe ("Multă sănătate", "vă dorim") și nu a sesizat răsturnarea ironică. A marcat `interpretation_problem: sentiment_vs_stance`, dar a rezolvat greșit conflictul.

- **Rândul 3** — "In acest moment mai putem spune doar Dumnezeu să îi apere pe cei care stau acolo..." — Modelul a marcat `target: unclear` și `stance: neutru`. De fapt, **ținta implicită este Primarul Negoiță / sistemul** (criticat în videoclip), iar stance-ul este `contra`. Modelul nu a folosit contextul din `video_title` ca ancoră.

- **Rândul 9** — "Vă mai dau niște firme din Galați care au alți administratori dar cu foști patroni" — Modelul a marcat `stance: neutru`. În realitate, comentariul completează un denunț al practicilor de evaziune și este implicit `contra` rețelei descrise în video. Lipsa unor cuvinte explicit negative l-a făcut pe model să clasifice neutru.

### A confundat sentimentul cu stance-ul?

Pe **8 din 10 cazuri** sentiment și stance au aceeași polaritate (`pro + pozitiv/mixt` sau `contra + negativ`). Singurele cazuri cu disociere clară:
- Rândul 0 (`stance: pro` + `sentiment: mixt`) — dar acolo modelul a clasificat greșit stance-ul (de fapt e contra)
- Rândul 2 ("Mulțumim Recorder", critică autoritățile) — `stance: contra` + `sentiment: negativ`, deși comentariul are și o componentă pozitivă (mulțumirea către Recorder)

**Concluzie:** modelul tratează cele două axe ca fiind aproape sinonime, deși în prompt am specificat explicit că sunt diferite. La rândul 0 a marcat `interpretation_problem: sentiment_vs_stance`, dar nu a folosit această observație ca să corecteze clasificarea — semn că a sesizat tensiunea fără să știe cum să o codifice.

### Au creat probleme sarcasmul, ambiguitatea sau țintele multiple?

- **Sarcasmul invers** (politețe sinceră interpretată ca... politețe sinceră) a apărut la rândul 0 și a stricat complet clasificarea. Modelul ignoră ironia când e ambalată în formule respectuoase românești ("Multă sănătate domnule Președinte").
- **Țintele implicite** au fost o problemă la rândurile 3 și 9 — modelul nu deduce ținta din contextul video, ci doar din textul comentariului.
- **Ambiguitate la rândul 2** — "Autoritățile abilitate să intervină" este țintă (`contra`) sau apel (`pro`)? Modelul a ales corect `contra`, dar prin formulare comentariul putea fi citit și ca cerere către autorități.
- **Țintă neconcret-instituțională la rândul 6** — "noua ordine mondiala" e o țintă conspirativă, fără referent real; modelul a clasificat corect ca `conspirativ` + `altul`, deci aici a mers.

### Ce aș schimba la versiunea următoare a promptului?

1. **Few-shot examples** pentru sarcasm românesc — un exemplu de tipul *"Multă sănătate, domnule X, să rămâneți alături de noi"* cu rezolvarea corectă (`stance: contra`, `tone: ironic`) ar ajuta modelul să recunoască tiparul ironic ambalat în politețe.
2. **Includerea contextului `video_title`** în prompt — multe comentarii capătă sens doar raportate la videoclipul comentat (ex. rândul 3, despre Primarul Negoiță).
3. **Forțarea unei distincții explicite** sentiment vs stance — adăugare în prompt: *"Dacă tonul comentariului contrazice poziționarea (ex: politețe ironică, compasiune față de critic), prioritizează stance-ul real al autorului, nu suprafața emoțională."*
4. **Eșantionare stratificată** pentru testare — `df.head(10)` a prins comentarii ușor de clasificat (RecorderRomania, ținte instituționale clare). Pentru evaluare reală ar trebui un sample care include explicit comentarii ironice, cu ținte multiple sau cu sentiment disociat de stance.
5. **Testare pe set mai mare** — 10 comentarii sunt prea puține pentru a calcula rate de eroare. Cu 100-200 s-ar putea calcula o rată de confuzie `sentiment vs stance` mai fiabilă.